## 1. Importing Data

---

In [1]:
# import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
import urllib

### 	🔌 Establishing Database Connection between SSMS 22 and Jupyter Notebook

In [2]:
# 1. Define your connection parameters
server = 'Aki-Dev\\SQLEXPRESS'       # Paste your SSMS Server Name here
database = 'OTTCustomerChurnDB'      # Your target database name
driver = 'ODBC Driver 17 for SQL Server' # The standard SQL Server driver

# 2. Build the connection string (with dynamic escaping for safety)
params = urllib.parse.quote_plus(
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;" # This enables Windows Authentication
)

# 3. Create the SQLAlchemy Engine/Connection
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# 4. Test the connection by running a query
sql_query = "SELECT name FROM sys.tables;"
tables = pd.read_sql(sql_query, engine)

# Display the tables in Jupyter from OTT Customer Churn Database
tables.head()

,name
0,db_customer
1,db_subscription
2,db_support


#### ➕ Create Dataframe for each table


In [3]:
for table_name in tables['name']:
    df=pd.read_sql(f"SELECT * FROM {table_name}", engine)
    globals()[f"df_{table_name}"] = df
    print(f"DataFrame df_{table_name} for table '{table_name}' has been created with shape: {df.shape}")

engine.dispose()  # Close the connection after fetching the data


DataFrame df_db_customer for table 'db_customer' has been created with shape: (21, 8)
DataFrame df_db_subscription for table 'db_subscription' has been created with shape: (21, 11)
DataFrame df_db_support for table 'db_support' has been created with shape: (9, 6)


#### 💡 Reading the Schema Before Reading the Data

Before pulling a single row, it is worth asking the database what it actually
contains. The cell below does that: it queries `INFORMATION_SCHEMA.COLUMNS`,
SQL Server's built-in catalogue, and lists the columns of every table. No
business data is touched, only the *metadata* that describes it.

> **Do we need this here? Honestly, no.**
> We have three small tables and we load all of them into pandas a moment later.
> At that point `df.info()` tells us everything this cell does, plus data types
> and null counts. I have kept it anyway, because this is the habit that saves
> you on a real database, and a small dataset is the cheapest place to learn it.

---

#### Where this earns its keep

| Situation | What the catalogue gives you |
|---|---|
| **Tables too big to load** | The structure of a 200-million-row table in milliseconds, so you select five columns instead of crashing the kernel with `SELECT *`. |
| **A database nobody documented** | A searchable map. "Which tables have a `customerid`?" becomes one query instead of an afternoon of clicking through SSMS. |
| **Working out the joins** | Column names shared across tables point straight at the join keys. |
| **Pipelines that run unattended** | A check that fails loudly when someone upstream renames or drops a column, rather than a dashboard that goes quietly wrong. |
| **Migrations** | A source-versus-target comparison that proves nothing was lost on the way. |
| **Privacy reviews** | A fast sweep for columns such as `dob`, `email` or `pincode` across the whole database. |
| **Documentation** | A data dictionary generated from the source of truth, not typed by hand and out of date within a month. |



#### Taking it further

The catalogue knows more than names. One query returns the full design of the
database, in column order:

```sql
SELECT TABLE_NAME, COLUMN_NAME, DATA_TYPE, IS_NULLABLE, CHARACTER_MAXIMUM_LENGTH
FROM INFORMATION_SCHEMA.COLUMNS
ORDER BY TABLE_NAME, ORDINAL_POSITION;
```

**Rule of thumb:** inspect the schema first, load data second. On three tables
it is a courtesy. On three hundred it is the difference between a five-minute
orientation and a lost day.

---

In [4]:
# Print table names and columns names

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

for table_name in tables['name']:
    print(f"\nTable: {table_name}")

    #Get column information
    columns_query = f"SELECT * FROM INFORMATION_SCHEMA.COLUMNS WHERE TABLE_NAME = '{table_name}'"
    df_columns = pd.read_sql(columns_query, engine)
    print(f"Columns: {list(df_columns['COLUMN_NAME'])}") # fetch the column names and save them in a list

engine.dispose()  # Close the connection after fetching the data


Table: db_customer
Columns: ['customerid', 'name', 'country', 'state', 'gender', 'dob', 'interests', 'pincode']

Table: db_subscription
Columns: ['customerid', 'subscription_start_date', 'subscription_type', 'renewal_date', 'plan_type', 'contract_type', 'cancellation_date', 'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score']

Table: db_support
Columns: ['customerid', 'complaint_date', 'escalations', 'csat_score', 'col_1', 'comment']


## 2. 🖌️ Data Cleaning

Explore each DataFrame to identify any inconsistencies or noise that could hinder the analysis.

The following commands provide different perspectives on the data.

---

In [5]:
# Display the first few rows of the DataFrame for the 'db_customer' table

df_db_customer.head()

,customerid,name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,chris,USA,New York,Male,1982-04-12,travel,None
1,0003-MKNFE,paul,USA,Kentucky,Male,1995-11-23,NaN,None
2,0004-TLHLJ,jenny,USA,Washington,Female,1978-02-15,movie,None
3,0011-IGKFF,mike,USA,Nevada,Male,2001-08-30,NaN,None
4,0013-EXCHZ,alex,USA,Washington,Female,1990-05-05,drama,None


In [6]:
# Display the summary information of the DataFrame for the 'db_customer' table
df_db_customer.info()

# Insight from the summary information of the DataFrame for the 'db_customer' table:
# a. drop columns - (interest and pincode) - not relevant to the analysis
# b. rename columns - name to customer_name
# c. fix missing values (Country)
# d. data standardization - gender (standardize the values to Male and Female).

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   customerid  21 non-null     str           
 1   name        21 non-null     str           
 2   country     20 non-null     str           
 3   state       21 non-null     str           
 4   gender      21 non-null     str           
 5   dob         21 non-null     datetime64[us]
 6   interests   4 non-null      str           
 7   pincode     0 non-null      object        
dtypes: datetime64[us](1), object(1), str(6)
memory usage: 1.4+ KB


In [7]:
# Display the last few rows of the DataFrame for the 'db_customer' table
df_db_customer.tail()

,customerid,name,country,state,gender,dob,interests,pincode
16,0020-JDNXP,riya,USA,Montana,Female,1994-08-19,NaN,None
17,0021-IKXGC,vasto,USA,Texas,Female,2000-09-02,NaN,None
18,0022-TCJCI,rocky,USA,Maine,Male,1983-12-30,NaN,None
19,0023-HGHWL,steve,USA,Utah,Men,1991-05-14,NaN,None
20,0023-UYUPN,pristina,USA,New York,Women,1977-10-06,NaN,None


#### a. Drop Column - 'interest' and 'pincode'

In [8]:
# Drop the columns 'interest' and 'pincode' from the DataFrame as they are not relevant to the analysis

df_db_customer.drop(columns=['interests', 'pincode'], inplace=True)


In [9]:
# Check df_db_customer

df_db_customer

,customerid,name,country,state,gender,dob
0,0002-ORFBO,chris,USA,New York,Male,1982-04-12
1,0003-MKNFE,paul,USA,Kentucky,Male,1995-11-23
2,0004-TLHLJ,jenny,USA,Washington,Female,1978-02-15
3,0011-IGKFF,mike,USA,Nevada,Male,2001-08-30
4,0013-EXCHZ,alex,USA,Washington,Female,1990-05-05
5,0013-MHZWF,christina,,Washington,Women,1988-12-10
6,0013-SMEOE,galabrielle,USA,Maine,Female,1976-09-21
7,0014-BMAQU,maddy,USA,Nevada,Male,1999-03-14
8,0015-UOCOJ,maya,NaN,Ontario,Women,1985-07-07
9,0016-QLJIS,arjun,Canada,Ontario,Male,1993-10-29


#### b. Rename Column - 'name -> customer_name'

In [10]:
df_db_customer.rename(columns={'name': 'customer_name'}, inplace=True)

# inplace = True, permanently modifies the DataFrame without needing to assign it back to df_db_customer.

In [11]:
df_db_customer


,customerid,customer_name,country,state,gender,dob
0,0002-ORFBO,chris,USA,New York,Male,1982-04-12
1,0003-MKNFE,paul,USA,Kentucky,Male,1995-11-23
2,0004-TLHLJ,jenny,USA,Washington,Female,1978-02-15
3,0011-IGKFF,mike,USA,Nevada,Male,2001-08-30
4,0013-EXCHZ,alex,USA,Washington,Female,1990-05-05
5,0013-MHZWF,christina,,Washington,Women,1988-12-10
6,0013-SMEOE,galabrielle,USA,Maine,Female,1976-09-21
7,0014-BMAQU,maddy,USA,Nevada,Male,1999-03-14
8,0015-UOCOJ,maya,NaN,Ontario,Women,1985-07-07
9,0016-QLJIS,arjun,Canada,Ontario,Male,1993-10-29


#### c. Fix missing values (Country)

In [12]:
# Check for missing values in the 'country' column of the DataFrame. In this column, we have some missing values (NaN) and some empty strings. 
# We will filter the DataFrame to show only those rows where the 'country' column is either NaN or an empty string.

df_db_customer[
    df_db_customer['country'].isna() | (df_db_customer['country'].str.strip() == '')]

# Data Reveals: There are 3 rows where the 'country' column is either NaN or an empty string.
# Solution: 
#  1. Replace the empty strings with NaN values for consistency.
#  2. Then, we can fill these NaN values with a existing pair value from a mapping dictionary, acting as a lookup.


,customerid,customer_name,country,state,gender,dob
5,0013-MHZWF,christina,,Washington,Women,1988-12-10
8,0015-UOCOJ,maya,NaN,Ontario,Women,1985-07-07
12,0018-NYROU,charlot,,Texas,Female,2004-12-01


In [13]:
# 1. Replace the empty strings with NaN values for consistency.


df_db_customer['country'] = (
    df_db_customer['country']
    .str.replace(r'^\s+|\s+$', '', regex=True)   # trim leading and trailing whitespace
    .replace('', pd.NA)                          # anything now empty becomes null
)    # trim leading and trailing whitespace

# store it back to country column of the DataFrame.



In [14]:
df_db_customer['country']

0        USA
1        USA
2        USA
3        USA
4        USA
5        NaN
6        USA
7        USA
8        NaN
9     Canada
10       USA
11       USA
12       NaN
13       USA
14       USA
15       USA
16       USA
17       USA
18       USA
19       USA
20       USA
Name: country, dtype: str

In [15]:
df_db_customer[
    df_db_customer['country'].isna() | (df_db_customer['country'].str.strip() == '')]

,customerid,customer_name,country,state,gender,dob
5,0013-MHZWF,christina,NaN,Washington,Women,1988-12-10
8,0015-UOCOJ,maya,NaN,Ontario,Women,1985-07-07
12,0018-NYROU,charlot,NaN,Texas,Female,2004-12-01


In [16]:
# 2. Fill the missing countries using the state column.
#
#    Approach:
#    a. Build a lookup dictionary of state -> country, using only the rows
#       where the country is present (e.g. {'Texas': 'USA'}).
#    b. For each row with a missing country, look up its state in the
#       dictionary and fill in the matching country.
#
#    Assumption: each state belongs to exactly one country. This was
#    checked and holds for this dataset.
#
# ------------------------------------------------------------------------------------

# a. Keep only rows that have a country, then map each state to its country

lookup_state_country = df_db_customer.dropna(subset=['country']).set_index('state')['country'].to_dict()

# b. Fill the missing countries from the lookup; existing values are left untouched

df_db_customer['country'] = df_db_customer['country'].fillna(df_db_customer['state'].map(lookup_state_country))

In [17]:
df_db_customer['country'] 

0        USA
1        USA
2        USA
3        USA
4        USA
5        USA
6        USA
7        USA
8     Canada
9     Canada
10       USA
11       USA
12       USA
13       USA
14       USA
15       USA
16       USA
17       USA
18       USA
19       USA
20       USA
Name: country, dtype: str

#### d. Data standardization - gender (standardize the values to Male and Female).

In [18]:
# df_db_customer['gender'].unique()

df_db_customer['gender'] = df_db_customer['gender'].replace({'Men': 'Male', 'Women': 'Female'})

In [19]:
df_db_customer['gender']

0       Male
1       Male
2     Female
3       Male
4     Female
5     Female
6     Female
7       Male
8     Female
9       Male
10      Male
11    Female
12    Female
13    Female
14      Male
15    Female
16    Female
17    Female
18      Male
19      Male
20    Female
Name: gender, dtype: str

In [20]:
df_db_subscription

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
0,0023-UYUPN,2022-12-31,Paid,2025-12-31,Standard,Monthly,None,NaN,13.99,790,47
1,0023-HGHWL,2020-06-23,Organic,2025-06-23,Premium,Annual,None,NaN,22.99,1955,7
2,0022-TCJCI,2023-09-14,Refferal,2024-09-14,Basic,Monthly,2024-09-14,Forgot to cancel trial,16.99,42,99
3,0021-IKXGC,2021-07-07,Paid,2025-07-07,Standard,Annual,None,NaN,13.99,840,27
4,0020-JDNXP,2022-01-19,Organic,2024-01-19,Premium,Annual,None,NaN,20.99,550,62
5,0020-INWCK,2023-05-06,Organic,2025-05-06,Standard,Monthly,None,NaN,13.99,840,27
6,0019-GFNTW,2019-11-11,Paid,2025-11-11,Premium,Annual,None,NaN,92.99,2185,3
7,0019-EFAEP,2022-08-25,Refferal,2024-08-25,Basic,Monthly,2024-10-31,Switched to competitor,12.99,160,76
8,0018-NYROU,2020-10-09,Organic,2025-10-09,Standard,Annual,None,NaN,13.99,980,19
9,0017-IUDMW,2023-03-17,Refferal,2024-03-17,Standard,Monthly,2024-05-01,Poor streaming quality,7.99,270,83


In [21]:
df_db_subscription.info()

# Insight from the summary information of the DataFrame for the 'db_subscription' table:

# - Fix date column (subscription_start_date,renewal_date,cancellation_date)- Ensure all date columns are in the correct datetime format

# - Fixing value in column subscription_type - 'Refferal' to 'Referral' - Correcting the spelling mistake in the subscription_type column.

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customerid               21 non-null     str    
 1   subscription_start_date  21 non-null     object 
 2   subscription_type        21 non-null     str    
 3   renewal_date             21 non-null     object 
 4   plan_type                21 non-null     str    
 5   contract_type            21 non-null     str    
 6   cancellation_date        6 non-null      object 
 7   cancellation_reason      6 non-null      str    
 8   monthly_charges          21 non-null     float64
 9   cltv                     21 non-null     int64  
 10  churn_score              21 non-null     int64  
dtypes: float64(1), int64(2), object(3), str(5)
memory usage: 1.9+ KB


In [22]:
# Fix date column (sub)- Ensure all date columns are in the correct datetime format

date_columns = ['subscription_start_date', 'renewal_date', 'cancellation_date']

df_db_subscription[date_columns] = df_db_subscription[date_columns].apply(pd.to_datetime)  # Convert to datetime

df_db_subscription.info()

<class 'pandas.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype        
---  ------                   --------------  -----        
 0   customerid               21 non-null     str          
 1   subscription_start_date  21 non-null     datetime64[s]
 2   subscription_type        21 non-null     str          
 3   renewal_date             21 non-null     datetime64[s]
 4   plan_type                21 non-null     str          
 5   contract_type            21 non-null     str          
 6   cancellation_date        6 non-null      datetime64[s]
 7   cancellation_reason      6 non-null      str          
 8   monthly_charges          21 non-null     float64      
 9   cltv                     21 non-null     int64        
 10  churn_score              21 non-null     int64        
dtypes: datetime64[s](3), float64(1), int64(2), str(5)
memory usage: 1.9 KB


In [23]:
# Fixing value in column subscription_type - 'Refferal' to 'Referral' - Correcting the spelling mistake in the subscription_type column.

df_db_subscription['subscription_type'] = df_db_subscription['subscription_type'].replace({'Refferal': 'Referral'})

df_db_subscription['subscription_type']

0         Paid
1      Organic
2     Referral
3         Paid
4      Organic
5      Organic
6         Paid
7     Referral
8      Organic
9     Referral
10     Organic
11     Organic
12     Organic
13     Organic
14    Referral
15        Paid
16    Referral
17        Paid
18     Organic
19        Paid
20    Referral
Name: subscription_type, dtype: str

In [24]:
df_db_support

,customerid,complaint_date,escalations,csat_score,col_1,comment
0,0022-TCJCI,2024-09-14,N,90,None,received refund
1,0022-TCJCI,2024-09-13,Y,10,None,NaN
2,0019-EFAEP,2024-09-27,Y,30,None,NaN
3,0017-IUDMW,2024-04-10,Y,25,None,NaN
4,0013-SMEOE,2024-11-01,N,30,None,NaN
5,0013-MHZWF,2025-03-18,N,90,None,guidance to renew
6,0013-EXCHZ,2024-01-20,Y,20,None,NaN
7,0003-MKNFE,2024-08-28,Y,10,None,demaned refund
8,0003-MKNFE,2024-08-28,N,60,None,service issue


In [25]:
df_db_support.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   customerid      9 non-null      str           
 1   complaint_date  9 non-null      datetime64[us]
 2   escalations     9 non-null      str           
 3   csat_score      9 non-null      int64         
 4   col_1           0 non-null      object        
 5   comment         4 non-null      str           
dtypes: datetime64[us](1), int64(1), object(1), str(3)
memory usage: 564.0+ bytes


In [ ]:
# Remove col_1 and comment columns from the DataFrame as they are not relevant to the analysis

df_db_support.drop(columns=['col_1', 'comment'], inplace=True)

df_db_support


,customerid,complaint_date,escalations,csat_score
0,0022-TCJCI,2024-09-14,N,90
1,0022-TCJCI,2024-09-13,Y,10
2,0019-EFAEP,2024-09-27,Y,30
3,0017-IUDMW,2024-04-10,Y,25
4,0013-SMEOE,2024-11-01,N,30
5,0013-MHZWF,2025-03-18,N,90
6,0013-EXCHZ,2024-01-20,Y,20
7,0003-MKNFE,2024-08-28,Y,10
8,0003-MKNFE,2024-08-28,N,60
